# Effect of Normalization in Simple Linear Regression

This notebook compares gradient-descent training on raw versus normalized input features.

The goal is to observe how input scale affects numerical stability and convergence speed, without changing the underlying linear relationship.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=4, suppress=True)

SEED = 42
rng = np.random.default_rng(SEED)

## Experiment Design

1. Generate synthetic data following a linear relationship, but with a wide input range so the effect of normalization becomes visible.
2. Train the same gradient-descent model twice — once on the raw feature, once on the min-max normalized feature — using the same learning rate.
3. Compare the loss curves, the recovered weights, and the fitted lines in the original scale.
4. Show that the unnormalized model can reach the same solution only if the learning rate is reduced significantly.

In [ ]:
# Synthetic data: wide x range to amplify scaling effects
n_samples = 60
true_slope = 4.0
true_intercept = -2.0

x_raw = rng.uniform(0, 100, size=n_samples)
noise = rng.normal(0, 12, size=n_samples)
y = true_slope * x_raw + true_intercept + noise

x_min = x_raw.min()
x_max = x_raw.max()
x_range = x_max - x_min
x_norm = (x_raw - x_min) / x_range   # min-max normalization -> [0, 1]

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x_raw, y, color="tab:blue", alpha=0.8)
ax.set_title("Raw data")
ax.set_xlabel("x (raw)")
ax.set_ylabel("y")
plt.show()

print(f"x min:   {x_min:.3f}")
print(f"x max:   {x_max:.3f}")
print(f"x range: {x_range:.3f}")

In [ ]:
def build_design_matrix(feature):
    """Return [1 | feature] design matrix (bias column first)."""
    X = np.column_stack([np.ones_like(feature), feature])
    return X.astype(float)


def linear_activation(X, W):
    return X @ W


def mse(X, Y, W):
    residuals = linear_activation(X, W) - Y
    return float(np.mean(residuals ** 2))


def mse_gradient(X, Y, W):
    n = len(Y)
    residuals = linear_activation(X, W) - Y
    return (2 / n) * (residuals.T @ X).T


def update_weights(X, Y, W, learning_rate):
    return W - learning_rate * mse_gradient(X, Y, W)


def train(epochs, X, Y, W_init, learning_rate):
    W = W_init.astype(float).copy()
    loss_history = [mse(X, Y, W)]

    for epoch in range(epochs):
        W = update_weights(X, Y, W, learning_rate)
        current_loss = mse(X, Y, W)
        loss_history.append(current_loss)

        if not np.isfinite(current_loss):
            print(f"Training stopped at epoch {epoch + 1}: loss is not finite.")
            break

    return {
        "weights": W,
        "loss_history": np.array(loss_history, dtype=float),
        "epochs_ran": len(loss_history) - 1,
    }


def to_original_scale(W_norm, x_min, x_range):
    """Convert weights learned on min-max normalized x back to the original x scale.

    If x_norm = (x - x_min) / x_range, then:
      y = W[0] + W[1]*x_norm
        = (W[0] - W[1]*x_min/x_range) + (W[1]/x_range)*x
    """
    slope = W_norm[1, 0] / x_range
    intercept = W_norm[0, 0] - W_norm[1, 0] * x_min / x_range
    return np.array([[intercept], [slope]])

In [ ]:
Y = y.reshape(-1, 1)
X_raw  = build_design_matrix(x_raw)
X_norm = build_design_matrix(x_norm)

shared_lr = 0.01
epochs    = 250
W0        = np.zeros((2, 1))

raw_run  = train(epochs, X_raw,  Y, W0, shared_lr)
norm_run = train(epochs, X_norm, Y, W0, shared_lr)

raw_weights  = raw_run["weights"]
norm_weights = norm_run["weights"]
norm_weights_orig = to_original_scale(norm_weights, x_min, x_range)

raw_cond  = np.linalg.cond(X_raw.T  @ X_raw)
norm_cond = np.linalg.cond(X_norm.T @ X_norm)

print(f"Shared learning rate:          {shared_lr}")
print(f"Condition number (raw):        {raw_cond:.2e}")
print(f"Condition number (normalized): {norm_cond:.2e}")
print()
print("Weights (raw training):",        raw_weights.ravel())
print(f"Final MSE (raw):               {raw_run['loss_history'][-1]:.3e}")
print()
print("Weights (normalized, original scale):", norm_weights_orig.ravel())
print(f"Final MSE (normalized):        {norm_run['loss_history'][-1]:.3e}")

In [ ]:
x_line = np.linspace(x_raw.min(), x_raw.max(), 200)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Loss curves ---
raw_hist  = raw_run["loss_history"]
norm_hist = norm_run["loss_history"]

axes[0].semilogy(np.arange(raw_hist.size)[np.isfinite(raw_hist)],
                 raw_hist[np.isfinite(raw_hist)],
                 label="Raw input", color="tab:red")
axes[0].semilogy(np.arange(norm_hist.size)[np.isfinite(norm_hist)],
                 norm_hist[np.isfinite(norm_hist)],
                 label="Normalized input", color="tab:green")
axes[0].set_title("Loss curve — same learning rate")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()

# --- Fitted lines in original scale ---
axes[1].scatter(x_raw, y, label="Data", color="tab:blue", alpha=0.7)

if np.all(np.isfinite(raw_weights)):
    axes[1].plot(x_line,
                 raw_weights[0, 0] + raw_weights[1, 0] * x_line,
                 label="Fit — raw input", color="tab:red", linewidth=2)

axes[1].plot(x_line,
             norm_weights_orig[0, 0] + norm_weights_orig[1, 0] * x_line,
             label="Fit — normalized input (original scale)", color="tab:green", linewidth=2)

axes[1].plot(x_line,
             true_intercept + true_slope * x_line,
             label="True relationship", color="black", linestyle="--", linewidth=1.5)

axes[1].set_title("Fitted lines in original scale")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Experiment 2: can the raw model converge with a much smaller learning rate?
raw_safe_lr     = 0.00001
raw_safe_epochs = 5000

raw_safe_run     = train(raw_safe_epochs, X_raw, Y, W0, raw_safe_lr)
raw_safe_weights = raw_safe_run["weights"]

print(f"Learning rate (raw, reduced): {raw_safe_lr}")
print("Weights (raw, reduced lr):", raw_safe_weights.ravel())
print(f"Final MSE (raw, reduced lr):  {raw_safe_run['loss_history'][-1]:.3f}")
print()
print("Weights (normalized, original scale):", norm_weights_orig.ravel())
print(f"Final MSE (normalized):               {norm_run['loss_history'][-1]:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(raw_safe_run["loss_history"],
            label=f"Raw input  (lr={raw_safe_lr}, {raw_safe_epochs} epochs)",
            color="tab:orange")
ax.semilogy(norm_run["loss_history"],
            label=f"Normalized (lr={shared_lr}, {epochs} epochs)",
            color="tab:green")
ax.set_title("Convergence after adjusting the learning rate")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.legend()
plt.show()

## Conclusions

- **Same learning rate**: the normalized model converges smoothly, while the raw model diverges or stalls because the large input scale causes overshooting.
- **Condition number**: min-max normalization squeezes $x$ into $[0, 1]$, dramatically lowering the condition number of $X^\top X$ and making the gradient-descent landscape more isotropic and easier to optimize.
- **Equivalent solutions**: both models learn the same underlying line — normalization affects *how easily* the optimizer finds the solution, not the solution itself.
- **Trade-off**: the raw model can converge with a much smaller learning rate, but requires far more epochs to reach a comparable loss, illustrating the practical benefit of normalizing inputs before training.